# 3단계 준비: 임베딩 입력 파일 생성

## 목적
2단계 결과(`reviews_step2_dedup.csv`)에서 임베딩과 이후 분석에 필요한 컬럼만 추려서 `embedding_input.parquet`로 저장한다.
이 파일은 Colab에 업로드해서 3단계 임베딩의 입력으로 사용한다.

## 입력 / 출력
- **입력**: `reviews_step2_dedup.csv` (2단계 결과)
- **출력**: `embedding_input.parquet` (Colab 업로드용)

## 처리 절차
1. 데이터 로드
2. 유효 리뷰 필터링 (`is_valid_for_topic = True`)
3. 필요 컬럼 선택
4. Parquet으로 저장

## 1. 데이터 로드

In [1]:
import pandas as pd
import numpy as np

INPUT_PATH = 'reviews_step2_dedup.csv'

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"로드 완료: {len(df):,}건")
print(f"컬럼 수: {df.shape[1]}개")

로드 완료: 623,665건
컬럼 수: 66개


## 2. 유효 리뷰 필터링

`is_valid_for_topic = True`인 리뷰만 임베딩 대상으로 선택.
무효 리뷰(한글 5자 미만)는 별도 보관해서 7단계 감성분석에서 평점 정보만 활용 가능.

In [2]:
# 필터링
mask_valid = df['is_valid_for_topic']

df_valid = df[mask_valid].copy().reset_index(drop=True)
df_invalid = df[~mask_valid].copy()

print(f"임베딩 대상 (한글 5자 이상): {len(df_valid):,}건")
print(f"제외 (한글 5자 미만):       {len(df_invalid):,}건")

# 무효 리뷰 별도 보관
df_invalid.to_csv('reviews_invalid_for_topic.csv', index=False)
print(f"\n무효 리뷰 저장: reviews_invalid_for_topic.csv")

임베딩 대상 (한글 5자 이상): 623,378건
제외 (한글 5자 미만):       287건

무효 리뷰 저장: reviews_invalid_for_topic.csv


## 3. 임베딩 입력에 필요한 컬럼 선택

### 컬럼 그룹
- **그룹 A** (필수): 임베딩 매칭용
- **그룹 B** (분석 필수): 토픽-메타데이터 매칭
- **그룹 C** (감성분석용): 항목별 점수 + 감성 강도 플래그
- **그룹 D** (추가 분석): 인구통계, 옵션 정보

In [3]:
# 그룹별 컬럼 정의
group_a = ['리뷰번호', '리뷰내용_clean']

group_b = ['goodsNo', '리뷰타입', '평점', '작성일']

group_c = [
    # 항목별 만족도 (원본 카테고리)
    '사이즈', '화면대비색감', '퀄리티', '구김', 
    '두께감', '신축성', '색감', '보온성',
    # 항목별 점수 (1~5)
    '퀄리티_점수', '보온성_점수', '신축성_점수', '두께감_점수', '구김_점수',
    # 편차 절대값 (0~2, 0에 가까울수록 이상적)
    '사이즈_편차절대', '화면대비색감_편차절대', '색감_편차절대',
    # 응답 여부
    '만족도_응답여부',
    # 감성 강도 플래그
    'laugh_count', 'cry_count', 'exclamation_count', 
    'question_count', 'emoji_count',
    'has_repetition',      # 추가
    'text_length_orig',    # 추가
]

group_d = [
    '성별', '구매사이즈', '구매상세', 
    '사진유무', '도움돼요', '체험단',
]

selected_cols = group_a + group_b + group_c + group_d

# 실제 존재하는 컬럼만 (오타나 누락 대비)
existing_cols = [c for c in selected_cols if c in df_valid.columns]
missing_cols = [c for c in selected_cols if c not in df_valid.columns]

print(f"선택된 컬럼: {len(existing_cols)}개")
if missing_cols:
    print(f"⚠️  누락된 컬럼 ({len(missing_cols)}개): {missing_cols}")

df_export = df_valid[existing_cols].copy()
print(f"\n[그룹별 컬럼 수]")
print(f"  A (필수):       {sum(1 for c in group_a if c in existing_cols)}개")
print(f"  B (분석 필수):  {sum(1 for c in group_b if c in existing_cols)}개")
print(f"  C (감성분석):   {sum(1 for c in group_c if c in existing_cols)}개")
print(f"  D (추가 분석):  {sum(1 for c in group_d if c in existing_cols)}개")

선택된 컬럼: 36개

[그룹별 컬럼 수]
  A (필수):       2개
  B (분석 필수):  4개
  C (감성분석):   24개
  D (추가 분석):  6개


## 4. Parquet으로 저장

`pyarrow` 또는 `fastparquet` 라이브러리가 필요하다.
설치되어 있지 않으면: `pip install pyarrow`

In [4]:
import os

OUTPUT_PATH = 'embedding_input.parquet'

# Parquet 저장 (pyarrow 백엔드)
df_export.to_parquet(OUTPUT_PATH, index=False, engine='pyarrow')

# 파일 크기 확인
size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"저장 완료: {OUTPUT_PATH}")
print(f"  - 행 수:   {len(df_export):,}")
print(f"  - 컬럼 수: {df_export.shape[1]}")
print(f"  - 파일 크기: {size_mb:.1f} MB")

# CSV로도 저장하면 비교 가능 (옵션)
csv_size_estimated_mb = os.path.getsize(INPUT_PATH) / (1024 * 1024) * (len(df_export) / len(df))
print(f"\n참고: 같은 데이터를 CSV로 저장하면 약 {csv_size_estimated_mb:.0f} MB (3~5배 큼)")

저장 완료: embedding_input.parquet
  - 행 수:   623,378
  - 컬럼 수: 36
  - 파일 크기: 49.2 MB

참고: 같은 데이터를 CSV로 저장하면 약 360 MB (3~5배 큼)


## 5. 저장 결과 검증

In [5]:
# Parquet 다시 로드해서 확인
df_check = pd.read_parquet(OUTPUT_PATH)

print(f"[로드 검증]")
print(f"  행 수: {len(df_check):,} (원본 {len(df_export):,}와 일치: {len(df_check) == len(df_export)})")
print(f"  컬럼: {df_check.shape[1]}개")

print(f"\n[샘플 (첫 3건)]")
display(df_check.head(3))

print(f"\n[데이터 타입]")
print(df_check.dtypes)

[로드 검증]
  행 수: 623,378 (원본 623,378와 일치: True)
  컬럼: 36개

[샘플 (첫 3건)]


,리뷰번호,리뷰내용_clean,goodsNo,리뷰타입,평점,작성일,사이즈,화면대비색감,퀄리티,구김,...,question_count,emoji_count,has_repetition,text_length_orig,성별,구매사이즈,구매상세,사진유무,도움돼요,체험단
0,34612047,요즘 입기 좋은 것 같아요 무난무난하게 잘 입고있습니다,1733275,goods,5.0,2022-11-07 18:13:53,NaN,NaN,NaN,NaN,...,0,0,False,30,여성,L,다크그레이,False,0,False
1,51564285,잠옷용으로 구매했어요 편하고 그냥 입기에도 조아요,3070563,goods,5.0,2023-11-23 15:50:39,NaN,NaN,NaN,NaN,...,0,0,False,27,missing,2XL,블랙,False,0,False
2,46679669,잠옷용으로 휘뚜루마뚜루 입으려고 좀 크게 사긴했는데 진짜 많이 커요 조금은 두꺼운 ...,3251750,goods,5.0,2023-08-01 15:28:26,NaN,NaN,NaN,NaN,...,0,0,False,52,여성,XL,블랙,False,0,False



[데이터 타입]
리뷰번호                   int64
리뷰내용_clean               str
goodsNo                int64
리뷰타입                     str
평점                   float64
작성일                      str
사이즈                      str
화면대비색감                   str
퀄리티                      str
구김                       str
두께감                      str
신축성                      str
색감                       str
보온성                      str
퀄리티_점수               float64
보온성_점수               float64
신축성_점수               float64
두께감_점수               float64
구김_점수                float64
사이즈_편차절대             float64
화면대비색감_편차절대          float64
색감_편차절대              float64
만족도_응답여부                 str
laugh_count            int64
cry_count              int64
exclamation_count      int64
question_count         int64
emoji_count            int64
has_repetition          bool
text_length_orig       int64
성별                       str
구매사이즈                    str
구매상세                     str
사진유무                    bool
도움돼요

## 6. 다음 단계: Google Drive 업로드

생성된 `embedding_input.parquet`을 Google Drive의 프로젝트 폴더에 업로드한다.

예시 폴더 구조:
```
MyDrive/
└── review_analysis/
    ├── embedding_input.parquet   ← 방금 만든 파일
    ├── embeddings.npy            ← Colab에서 생성될 예정
    └── tokenized_docs.pkl        ← Colab에서 생성될 예정
```

업로드 완료되면 Colab 노트북(`step3_colab_embedding_bertopic.ipynb`)을 실행한다.